In [1]:
from ema_workbench.connectors.vensim import VensimModel
from ema_workbench.analysis.plotting import lines, Density, kde_over_time
from ema_workbench.analysis import pairs_plotting, prim, feature_scoring, dimensional_stacking, regional_sa, get_ex_feature_scores, RuleInductionType, multiple_densities
from numpy.lib import recfunctions as rf
import itertools

from ema_workbench import SequentialEvaluator, Policy
from ema_workbench.em_framework.samplers import FullFactorialSampler

from ema_workbench.analysis.plotting import kde_over_time

import ema_workbench.analysis.cart as cart

import math
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

from ema_workbench import (
    Model,
    RealParameter,
    Constant,
    ScalarOutcome,
    TimeSeriesOutcome,
    ema_logging,
    perform_experiments,
    MultiprocessingEvaluator,
    SequentialEvaluator,
    CategoricalParameter,
    save_results
)

C:\Users\31628\Documents\Studie\Python\Lib\site-packages\ema_workbench\connectors\__init__.py:29: ImportWarning: netlogo connector not available
  warnings.warn("netlogo connector not available", ImportWarning)
C:\Users\31628\Documents\Studie\Python\Lib\site-packages\ema_workbench\connectors\__init__.py:34: ImportWarning: simio connector not available
  warnings.warn("simio connector not available", ImportWarning)


In [2]:
# if __name__ == "__main__":
# Dit stond op ema workbench

#from ema_workbench.util import ema_logging
#ema_logging.log_to_stderr(ema_logging.DEBUG)

# instantiate a model
wd = "../models/"
model = VensimModel("LabourMarketModel", wd=wd, model_file="Merijn_Simulation_model14.2.vpmx")

#print("Attempting to run model setup...")
#try:
#    model.run_model()
#except Exception as e:
#    print("Model setup failed:", e)


model.uncertainties = [
    RealParameter("female fertility rate", 1.3, 1.8),
    RealParameter("Exogenous Healthcare level increase", 0.00, 0.04),
    #RealParameter("annual change in FTE per age due to policy", 0.005, 0.015),
    RealParameter("annual change in FTE per age due to higher wages", 0.0005, 0.0015),
    RealParameter("factor extra change in FTE per age if retired", 1.5, 3),
    RealParameter("wages threshold for decreasing FTE per worker", 4500, 7000),
    RealParameter("exogenous trend increase in Male Labour Participation", 0.005, 0.02),
    RealParameter("exogenous trend increase in Female Labour Participation", 0.005, 0.02),
    #RealParameter("annual change in labour participation due to policy", 0.005, 0.02),
    RealParameter("annual change in labour participation due to higher wages", 0.0005, 0.002),
    RealParameter("factor extra change in labour participation if retired", 1.5, 3),
    RealParameter("wages threshold for decreasing labour participation", 5000, 8000),
    RealParameter("average hiring time for new workforce", 0.15, 0.75),
    RealParameter(r"average time working at same job[Industry sector]", 7.5, 15),
    RealParameter(r"average time working at same job[Service sector]", 6, 12),
    #RealParameter(r"GDP Scenario 1 exogenous gap[Industry sector]", 6400000000, 9600000000),
    #RealParameter(r"GDP Scenario 1 exogenous gap[Service sector]", 20719600000, 31079400000),
    RealParameter("exogenous trend in capital", 0.001, 0.01),
    RealParameter("Wages on capital coefficient", 5, 15),
    RealParameter("return to baseline of capital delay", 1, 5),
    RealParameter("exogenous trend in technology", 0.001, 0.01),
    RealParameter("Wages on technology coefficient", 1, 5),
    RealParameter("exogenous trend in human capital", 0.005, 0.02),
    RealParameter("capital on productivity coefficient", 130, 200),
    RealParameter("technology on productivity coefficient", 2500, 3500),
    RealParameter("human capital on productivity coefficient", 2500, 3500),
    RealParameter("wages on productivity coefficient", 1, 5),
    #RealParameter("max retirement age", 70, 80),
    #RealParameter("M 2 Policy Goal lower percentage", 0.5, 0.85),
    #RealParameter("decreasing factor migration policy 2", 0.005, 0.025),
    #RealParameter("M 4 Policy Goal higher percentage", 1.3, 1.75),
    #RealParameter("increasing factor migration policy 4", 0.005, 0.02),
    #RealParameter("labour balance on migration coefficient", 0.001, 0.01),
    #RealParameter("delay factor adapting migration policy", 2.5, 10),
    #RealParameter("labour balance and wages coefficient", 100, 160),
    #RealParameter("exogenous trend in wages in Service", 5, 25),
    #RealParameter("exogenous trend in wages in Industry", 5, 25),
    CategoricalParameter("Scenario Switch Healthcare scenario's", (1, 2))
    #CategoricalParameter("Scenario Switch GDP scenario's", (1, 2, 3))
                        ]
    
    #Old example:
    #RealParameter("annual change in FTE per age", 0.002, 0.05),
    #RealParameter("annual change in labour participation", 0.01, 0.1),
    #RealParameter("delay factor change in percentage of workers per sector", 5, 15),
    
    #CategoricalParameter("Scenario Switch: Healthcare scenario's", (1, 2))



model.outcomes = [
        TimeSeriesOutcome("Total labour balance"),
        TimeSeriesOutcome("Unemployment rate"),
        TimeSeriesOutcome("Labour dependency ratio"),
        TimeSeriesOutcome("Total GDP"),
        TimeSeriesOutcome(r"Labour balance[Industry sector]"),
        TimeSeriesOutcome(r"Labour balance[Service sector]"),
        TimeSeriesOutcome(r"GDP[Industry sector]"),
        TimeSeriesOutcome(r"GDP[Service sector]"),            
        TimeSeriesOutcome(r"Productivity per sector[Industry sector]"),
        TimeSeriesOutcome(r"Productivity per sector[Service sector]"),
        TimeSeriesOutcome(r"Wages[Industry sector]"),
        TimeSeriesOutcome(r"Wages[Service sector]"),
        TimeSeriesOutcome("Total population")
                        ]

model.levers = [
    CategoricalParameter("Policy Switch Migration", (1, 3, 5)),
    #CategoricalParameter("Policy Switch Targeting ages in migration", (1, 2, 3)),
    CategoricalParameter("Policy Switch Retirement age", (0, 3)),
    CategoricalParameter("Policy Switch Wages scenario's", (1, 2))
                         ]

# Generate all 12 unique combinations of the levers
migration = [1, 3, 5]
retirement = [0, 3]
wages = [1, 2]

policy_combinations = list(itertools.product(migration, retirement, wages))

# Create a list of policies
policies = []
for i, (mig, ret, wag) in enumerate(policy_combinations):
    policies.append(
        Policy(f"policy_{i+1}", **{
            "Policy Switch Migration": mig,
            "Policy Switch Retirement age": ret,
            "Policy Switch Wages scenario's": wag
        })
    )

ema_logging.log_to_stderr(level=ema_logging.INFO)
  
#results = perform_experiments(model, 10, policies = 5)

with MultiprocessingEvaluator(model, n_processes=-1) as evaluator:
    results = evaluator.perform_experiments(1000, policies = policies)

 
#with SequentialEvaluator(model) as evaluator:
#    results = evaluator.perform_experiments(10)

#,  policies = 1

[MainProcess/INFO] pool started with 7 workers
[MainProcess/INFO] performing 1000 scenarios * 12 policies * 1 model(s) = 12000 experiments
100%|██████████████████████████████████| 12000/12000 [9:52:54<00:00,  2.96s/it]
[MainProcess/INFO] experiments finished
[MainProcess/INFO] terminating pool


In [4]:
from ema_workbench import save_results
save_results(results, '../results/New_Modelresults6.tar.gz')

C:\Users\31628\Documents\Studie\Python\Lib\site-packages\ema_workbench\em_framework\outcomes.py:541: UserWarning: still to be tested!!
  warnings.warn("still to be tested!!")
[MainProcess/INFO] results saved successfully to C:\Users\31628\Documents\Studie\Master Thesis\Models\EMA Workbench\results\New_Modelresults6.tar.gz


In [12]:
# unpack the results
experiments, outcomes = results

# fast check
print(outcomes.keys())

dict_keys(['TIME', 'Total labour balance', 'Unemployment rate', 'Labour dependency ratio', 'Total GDP', 'Labour balance[Industry sector]', 'Labour balance[Service sector]', 'GDP[Industry sector]', 'GDP[Service sector]', 'Productivity per sector[Industry sector]', 'Productivity per sector[Service sector]', 'Wages[Industry sector]', 'Wages[Service sector]'])


In [7]:
#experiments

In [9]:
#outcomes

In [22]:
#import pickle
#import os

# Print where the file will be saved
#print("File will be saved in:", os.getcwd())

#with open('results_backup.pkl', 'wb') as f:
#    pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL)



Bestand wordt opgeslagen in: C:\Users\31628\Documents\Studie\Master Thesis\Models\EMA Workbench\scripts
